In [1]:
import numpy as np 

In [118]:
def find_duplicate_indices(arr):
    """ 
    This function takes in an array and finds the indices of the duplicate entries.
    It returns a list of lists, where each list contains the indices of the duplicate entries.
    """

    # Find unique elements and their inverse indices
    _, inverse_indices = np.unique(arr, return_inverse=True)

    # Find indices of duplicate entries
    duplicate_indices = np.where(np.bincount(inverse_indices) > 1)[0]

    # Map back to original indices
    duplicates = [np.where(inverse_indices == index)[0] for index in duplicate_indices]

    return duplicates

def lattice_sym_constraint(lengths, angles, target_lengths, target_angles):
    """ 
    This function takes in the target lengths and angles and figures out the symmetry with respect to the lattice. 
    It then applies those symmetry constrains to the lengths and angles of the lattice."""

    # First, we need to figure out the symmetry of the lattice.
    # We will do this by finding the number of lengths and angles that are the same.
    # We will also look for 90 degree and 120 degree angles.

    #get the symmetry information 
    #this does not have to be differentiable
    same_lengths = find_duplicate_indices(target_lengths)
    same_angles = find_duplicate_indices(target_angles)
    angles_90 = np.where(np.isclose(target_angles, 90, atol=1e-3))[0]
    angles_120 = np.where(np.isclose(target_angles, 120, atol=1e-3))[0]
    
    #enforce duplicity and certain angles on the lengths. this must be differentiable    
    lengths[same_lengths[0][1:]] = lengths[same_lengths[0][0]]
    angles[same_angles[0][1:]] = angles[same_angles[0][0]]
    angles[angles_90] = 90
    angles[angles_120] = 120
    
    return lengths, angles

In [119]:
lengths = np.array([2, 3, 4])
angles = np.array([30, 60, 70])
target_lengths = np.array([2, 2, 4])
target_angles = np.array([90, 90, 90])

In [120]:
lattice_sym_constraint(lengths, angles, target_lengths, target_angles)

(array([2, 2, 4]), array([90, 90, 90]))

In [34]:
import pandas as pd
from pymatgen.core.structure import Structure   

In [35]:
train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20_final/train.csv')

In [38]:
from pymatgen.core.structure import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from io import StringIO

def get_crystal_structure_and_lattice_system(cif_string):
    # Use StringIO to create a file-like object from the CIF string
    cif_file = StringIO(cif_string)
    
    # Read the crystal structure directly from the file-like object
    structure = Structure.from_str(cif_file.getvalue(), fmt="cif")
    
    # Get the lattice system using SpacegroupAnalyzer
    analyzer = SpacegroupAnalyzer(structure)
    lattice_system = analyzer.get_lattice_type()
    
    # No need to clean up a temporary file, as StringIO is in-memory
    return structure, lattice_system



In [50]:
get_crystal_structure_and_lattice_system(train_df['cif'][5])

(Structure Summary
 Lattice
     abc : 4.424795640000001 4.424795639999999 7.67194038
  angles : 106.76071032 73.23928968 119.99999999999997
  volume : 122.6598362639547
       A : 4.236819132879253 0.0 1.2760016030676882
       B : -1.9262630480296168 3.7736119348082613 -1.2760016030676877
       C : 0.0 0.0 7.67194038
     pbc : True True True
 PeriodicSite: Mg0 (Mg) (-1.893, 3.753, 6.285) [0.005417, 0.9946, 0.9838]
 PeriodicSite: Mg1 (Mg) (1.119, 1.909, 3.957) [0.4941, 0.5059, 0.5177]
 PeriodicSite: Te2 (Te) (3.474, 0.4669, 3.808) [0.8763, 0.1237, 0.3712]
 PeriodicSite: Se3 (Se) (0.3801, 2.361, 6.41) [0.3742, 0.6258, 0.8773],
 'rhombohedral')